<a href="https://colab.research.google.com/github/ksmorozov7/Portfolio/blob/main/python%2Bsql.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Задание:**

Задание выполняется на Python. Требуемые библиотеки: pandas, random, datetime, sqlalchemy актуальной версии

1. Необходимо сгенерировать исходный набор данных (Регион, Дата продажи, Сумма продаж) из 10000 строк.
2. Сохранить полученный dataframe в Excel с названием pre_sales_data.xlsx
3. Загрузить Excel файл обратно в Python
4. Преобразовать dataframe в SQL-таблицу
5. Обработать данные с помощью SQL-запроса. Запрос должен в себя включать:
•	Фильтр по дате (значения для фильтрации подбираются на ваше усмотрение в зависимости от сгенерированных данных)
•	Группировку общих продаж по каждому региону и дате
•	Условный столбец, показывающий значение "Высокий"/"Средний"/"Низкий" в зависимости от суммы продаж в каждом регионе (значения для условия подбираются на ваше усмотрение в зависимости от сгенерированных данных).
•	Сортировку сгруппированных данных по продажам в убывающем порядке
6. Сохранить итоговый dataframe в новый Excel файл sales_data.xlsx



**Этап 0: Установка библиотек**

In [ ]:
!pip install pandas sqlalchemy openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 57.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.9/250.9 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 614.2/614.2 kB 45.6 MB/s eta 0:00:00


**Этап 1: Генерация данных и сохранение в Excel**

In [ ]:
import pandas as pd
import random
from datetime import datetime, timedelta
from google.colab import files

def generate_data(num_rows=10000):
    regions = ['Москва', 'Санкт-Петербург', 'Вологодская область', 'Ярославская область']

    start_date = datetime(2021, 1, 1)
    end_date = datetime(2024, 12, 31)

    data = []
    for _ in range(num_rows):
        region = random.choice(regions)
        date = start_date + timedelta(days=random.randint(0, (end_date - start_date).days))
        amount = round(random.uniform(1000, 500000), 2)
        data.append([region, date, amount])

    return pd.DataFrame(data, columns=['Регион', 'Дата продажи', 'Сумма продаж'])

# Генерация и сохранение данных
sales_data = generate_data()
sales_data.to_excel('pre_sales_data.xlsx', index=False)
files.download('pre_sales_data.xlsx')

print("Файл pre_sales_data.xlsx сгенерирован и скачан")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Файл pre_sales_data.xlsx сгенерирован и скачан


**Этап 2: Загрузка данных и преобразование в SQL**

In [ ]:
from sqlalchemy import create_engine
from google.colab import files

# Загрузка файла
print("Пожалуйста, загрузите файл pre_sales_data.xlsx")
uploaded = files.upload()
file_name = next(iter(uploaded))
loaded_data = pd.read_excel(file_name)

# Создание SQL-таблицы
engine = create_engine('sqlite:///:memory:')
loaded_data.to_sql('sales', engine, if_exists='replace', index=False)

print("Данные успешно загружены и преобразованы в SQL-таблицу")

Пожалуйста, загрузите файл pre_sales_data.xlsx


Saving pre_sales_data (1).xlsx to pre_sales_data (1).xlsx
Данные успешно загружены и преобразованы в SQL-таблицу


**Этап 3: Обработка данных SQL-запросом**

In [ ]:
# SQL-запрос
sql_query = """
WITH filtered_sales AS (
    SELECT
        date(`Дата продажи`) as `Дата`,
        `Регион`,
        `Сумма продаж`
    FROM sales
    WHERE date(`Дата продажи`) BETWEEN '2022-01-01' AND '2024-09-30'
),
region_stats AS (
    SELECT
        `Регион`,
        `Дата`,
        SUM(`Сумма продаж`) as `Общая сумма продаж`,
        CASE
            WHEN SUM(`Сумма продаж`) > 1000000 THEN 'Высокий'
            WHEN SUM(`Сумма продаж`) > 500000 THEN 'Средний'
            ELSE 'Низкий'
        END as `Уровень продаж`
    FROM filtered_sales
    GROUP BY `Регион`, `Дата`
)
SELECT * FROM region_stats
ORDER BY `Общая сумма продаж` DESC
"""

# Выполнение запроса
result_df = pd.read_sql_query(sql_query, engine)
print("Данные успешно обработаны SQL-запросом")

Данные успешно обработаны SQL-запросом


**Этап 4: Сохранение и скачивание результата**

In [ ]:
# Сохранение и скачивание
result_df.to_excel('sales_data.xlsx', index=False)
files.download('sales_data.xlsx')

print("Файл sales_data.xlsx с результатами доступен для скачивания")
display(result_df.head())

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Файл sales_data.xlsx с результатами доступен для скачивания


,Регион,Дата,Общая сумма продаж,Уровень продаж
0,Санкт-Петербург,2023-03-06,2638419.27,Высокий
1,Москва,2022-01-28,2425957.07,Высокий
2,Вологодская область,2024-04-13,2316464.62,Высокий
3,Москва,2024-03-24,2221238.91,Высокий
4,Москва,2022-05-10,2221169.70,Высокий
